# SolarSDE Notebook 1 — Data & CS-VAE Training

**Runtime:** ~4-6 hours on Colab T4 / Kaggle P100 (well under 8hr limit)

**This notebook:**
1. Downloads NREL CloudCV (8 days of sky images + irradiance) and BMS (90 days of meteorological data)
2. Preprocesses, aligns, filters, and splits into train/val/test parquet files
3. Trains the Cloud-State VAE (CS-VAE) for 20 epochs at 128×128
4. Extracts latent representations and computes CTI for all splits
5. Saves all outputs to Google Drive (Colab) or /kaggle/working (Kaggle)

**Outputs** (saved to `solarsde_outputs/` in persistent storage):
- `splits/{train,val,test}.parquet` — preprocessed data with metadata
- `checkpoints/vae_best.pt`, `vae_final.pt`
- `latents/{train,val,test}_{latents,cti,ghi,covariates,is_ramp}.npy`
- `results/vae_training_history.csv`
- Everything zipped for download at the end

**After this notebook runs**, proceed to Notebook 2 (SDE + Score + Main Evaluation).


In [ ]:
# ==== Install dependencies ====
import subprocess, sys
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

pip_install("pvlib", "properscoring", "pyarrow", "tqdm")
print("Dependencies installed.")


In [ ]:
# ==== Environment Detection & Setup ====
import os, sys, time, json, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

print(f"Environment: {'Colab' if IN_COLAB else 'Kaggle' if IN_KAGGLE else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception as e:
        print(f"Drive mount issue: {e}")
    PERSIST_DIR = Path("/content/drive/MyDrive/solarsde_outputs")
    WORK_DIR = Path("/content/solarsde")
elif IN_KAGGLE:
    PERSIST_DIR = Path("/kaggle/working/solarsde_outputs")
    WORK_DIR = Path("/kaggle/working/solarsde")
else:
    PERSIST_DIR = Path.cwd() / "solarsde_outputs"
    WORK_DIR = Path.cwd() / "solarsde_work"

PERSIST_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR = WORK_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PERSIST_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = PERSIST_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
LATENT_DIR = PERSIST_DIR / "latents"
LATENT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Persistent storage: {PERSIST_DIR}")
print(f"Working directory:  {WORK_DIR}")


In [ ]:
# ==== GPU Check ====
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device:          {torch.cuda.get_device_name(0)}")
    print(f"Memory:          {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
    print("WARNING: Running on CPU — will be slow. Change Runtime > Change runtime type > GPU")
print(f"Using device: {DEVICE}")


## 1. Download CloudCV dataset
The 8-day subset available on NREL's OEDI portal (~2.6 GB total).

In [ ]:
# ==== Download CloudCV dataset ====
import requests
from tqdm import tqdm

CLOUDCV_DIR = DATA_DIR / "cloudcv"
CLOUDCV_DIR.mkdir(parents=True, exist_ok=True)

CLOUDCV_FILES = {
    "README.md": "https://data.nlr.gov/system/files/248/1727758900-README.md",
    "2019_09_07.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_07.tar.gz",
    "2019_09_08.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_08.tar.gz",
    "2019_09_14.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_14.tar.gz",
    "2019_09_15.tar.gz": "https://data.nlr.gov/system/files/248/1727737056-2019_09_15.tar.gz",
    "2019_09_21.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_21.tar.gz",
    "2019_09_22.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_22.tar.gz",
    "2019_09_28.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_28.tar.gz",
    "2019_09_29.tar.gz": "https://data.nlr.gov/system/files/248/1727737586-2019_09_29.tar.gz",
}

def download_file(url, dest):
    if dest.exists() and dest.stat().st_size > 1000:
        print(f"  Already have: {dest.name} ({dest.stat().st_size/1e6:.1f} MB)")
        return True
    print(f"  Downloading {dest.name} ...")
    r = requests.get(url, stream=True, timeout=600, allow_redirects=True)
    r.raise_for_status()
    total = int(r.headers.get("content-length", 0))
    with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=dest.name) as pbar:
        for chunk in r.iter_content(chunk_size=65536):
            f.write(chunk); pbar.update(len(chunk))
    return True

print("=" * 70)
print("Downloading CloudCV dataset (8 days of sky images + irradiance)")
print("Expected total: ~2.6 GB")
print("=" * 70)
for name, url in CLOUDCV_FILES.items():
    download_file(url, CLOUDCV_DIR / name)
print("CloudCV download complete.")


In [ ]:
# ==== Extract CloudCV archives ====
import tarfile

print("Extracting tar.gz archives...")
for tgz in sorted(CLOUDCV_DIR.glob("2019_*.tar.gz")):
    stem = tgz.stem.replace(".tar", "")   # 2019_09_07
    out = CLOUDCV_DIR / stem
    if out.exists() and any(out.rglob("*.jpg")):
        n = sum(1 for _ in (out / "images").glob("*.jpg"))
        print(f"  {stem}: already extracted ({n} images)")
        continue
    out.mkdir(parents=True, exist_ok=True)
    with tarfile.open(tgz, "r:gz") as tf:
        tf.extractall(out)
    n_imgs = sum(1 for _ in (out / "images").glob("*.jpg"))
    print(f"  {stem}: extracted ({n_imgs} images)")

# Free disk by removing tarballs after extraction
for tgz in CLOUDCV_DIR.glob("*.tar.gz"):
    try:
        tgz.unlink()
    except Exception:
        pass
print("Extraction complete. tar.gz archives removed to free disk.")


## 2. Download BMS meteorological data
90 days of 1-minute resolution data from NREL SRRL via MIDC API.

In [ ]:
# ==== Download BMS meteorological data (90-day period) ====
BMS_DIR = DATA_DIR / "bms"
BMS_DIR.mkdir(parents=True, exist_ok=True)
BMS_PATH = BMS_DIR / "bms_srrl_2019.csv"

BMS_URL = "https://midcdmz.nrel.gov/apps/data_api.pl?site=BMS&begin=20190905&end=20191203&inst=1&type=data"

if BMS_PATH.exists() and BMS_PATH.stat().st_size > 10_000_000:
    print(f"BMS data already cached: {BMS_PATH.stat().st_size/1e6:.1f} MB")
else:
    print(f"Downloading BMS 1-minute data from NREL MIDC API ...")
    r = requests.get(BMS_URL, timeout=600)
    r.raise_for_status()
    BMS_PATH.write_text(r.text)
    print(f"Saved: {BMS_PATH.stat().st_size/1e6:.1f} MB")


## 3. Preprocessing
Parse CloudCV + BMS, align timestamps, interpolate BMS GHI to 10s, compute solar geometry, filter nighttime, detect ramp events, create chronological splits.

In [ ]:
# ==== Preprocessing: parse CloudCV + BMS, align, compute features ====
import numpy as np
import pandas as pd
from datetime import datetime
import pvlib
from pvlib.location import Location

SRRL = Location(latitude=39.742, longitude=-105.18, tz="America/Denver",
                altitude=1829, name="NREL SRRL")

def parse_ts(s):
    s = s.strip()
    if s.startswith("UTC-7_"):
        s = s[6:]
    date_p, time_p = s.split("-")
    y, mo, d = date_p.split("_")
    tp = time_p.split("_")
    h, mi, sec = tp[0], tp[1], tp[2]
    us = tp[3] if len(tp) > 3 else "0"
    return datetime(int(y), int(mo), int(d), int(h), int(mi), int(sec), int(us))

def load_cloudcv_day(day_dir):
    csv = day_dir / "pyranometer.csv"
    imgs = day_dir / "images"
    if not csv.exists():
        return pd.DataFrame()
    rows = []
    for line in open(csv):
        line = line.strip()
        if not line or "," not in line:
            continue
        parts = line.split(",")
        if len(parts) < 2:
            continue
        try:
            ts = parse_ts(parts[0])
        except Exception:
            continue
        mv = float(parts[1].strip())
        img_name = parts[0].strip() + ".jpg"
        img_path = imgs / img_name
        rows.append({
            "timestamp": ts, "millivolts": mv,
            "image_path": str(img_path),
            "image_exists": img_path.exists(),
        })
    df = pd.DataFrame(rows)
    if len(df) > 0:
        df["timestamp"] = pd.to_datetime(df["timestamp"])
    return df

print("Loading CloudCV days ...")
day_dirs = sorted([d for d in CLOUDCV_DIR.iterdir() if d.is_dir() and d.name.startswith("2019")])
all_dfs = []
for d in day_dirs:
    df = load_cloudcv_day(d)
    if len(df) > 0:
        all_dfs.append(df)
        print(f"  {d.name}: {len(df)} rows ({df['image_exists'].sum()} images)")

cloudcv = pd.concat(all_dfs, ignore_index=True).sort_values("timestamp").reset_index(drop=True)
print(f"Total CloudCV rows: {len(cloudcv)}")

print("\nLoading BMS ...")
bms_raw = pd.read_csv(BMS_PATH)
print(f"  BMS rows: {len(bms_raw)}")
print(f"  BMS columns: {len(bms_raw.columns)}")

# Build BMS timestamps from Year, DOY, MST
ts_list = []
for _, r in bms_raw.iterrows():
    try:
        y, doy, mst = int(r["Year"]), int(r["DOY"]), int(r["MST"])
        dt = datetime.strptime(f"{y}-{doy}", "%Y-%j").replace(hour=mst // 60, minute=mst % 60)
        ts_list.append(dt)
    except Exception:
        ts_list.append(pd.NaT)
bms_raw["timestamp"] = pd.to_datetime(ts_list)

bms = pd.DataFrame({
    "timestamp": bms_raw["timestamp"],
    "ghi_bms": bms_raw.get("Global LI-200 [W/m^2]"),
    "dni_bms": bms_raw.get("Direct NIP [W/m^2]"),
    "dhi_bms": bms_raw.get("Diffuse CM22-1 (vent/cor) [W/m^2]"),
    "temperature": bms_raw.get("Deck Dry Bulb Temp [deg C]"),
    "humidity": bms_raw.get("Deck RH [%]"),
    "wind_speed": bms_raw.get("Avg Wind Speed @ 19ft [m/s]"),
    "pressure": bms_raw.get("Station Pressure [mBar]"),
    "cloud_cover_total": bms_raw.get("Total Cloud Cover [%]"),
})
for c in bms.columns:
    if c != "timestamp":
        bms[c] = pd.to_numeric(bms[c], errors="coerce").replace([-7999, -6999, -9999], np.nan)

print(f"  BMS GHI range: [{bms['ghi_bms'].min():.1f}, {bms['ghi_bms'].max():.1f}] W/m²")

# Interpolate BMS GHI to 10s
print("\nInterpolating BMS GHI to 10-second resolution ...")
bms_ghi = bms[["timestamp", "ghi_bms"]].dropna().copy()
bms_ghi = bms_ghi.set_index("timestamp").sort_index()
bms_10s = bms_ghi.resample("10s").interpolate(method="linear")

cloudcv["ts_round"] = cloudcv["timestamp"].dt.round("10s")
ghi_vals = []
for ts in cloudcv["ts_round"]:
    if ts in bms_10s.index:
        ghi_vals.append(float(bms_10s.loc[ts, "ghi_bms"]))
    else:
        i = bms_10s.index.get_indexer([ts], method="nearest")[0]
        ghi_vals.append(float(bms_10s.iloc[i]["ghi_bms"]) if 0 <= i < len(bms_10s) else np.nan)
cloudcv["ghi"] = np.clip(ghi_vals, 0, None)

# Merge meteo covariates
cloudcv["ts_minute"] = cloudcv["timestamp"].dt.floor("min")
bms["ts_minute"] = bms["timestamp"].dt.floor("min")
merged = cloudcv.merge(bms.drop(columns=["timestamp"]), on="ts_minute", how="left")
for c in ["temperature", "humidity", "wind_speed", "pressure", "cloud_cover_total"]:
    if c in merged.columns:
        merged[c] = merged[c].ffill().fillna(0)

# Solar geometry + clear sky
print("\nComputing solar geometry + clear-sky ...")
tz_ts = pd.DatetimeIndex(merged["timestamp"]).tz_localize("America/Denver")
solpos = SRRL.get_solarposition(tz_ts)
merged["solar_zenith"] = solpos["apparent_zenith"].values
cs = SRRL.get_clearsky(tz_ts, model="ineichen")
merged["ghi_clearsky"] = cs["ghi"].values
with np.errstate(divide="ignore", invalid="ignore"):
    kt = merged["ghi"].values / merged["ghi_clearsky"].values
    kt = np.where(merged["ghi_clearsky"].values < 10, 0.0, kt)
merged["clear_sky_index"] = np.clip(kt, 0, 1.5)

# Quality filter: daytime, image exists, valid GHI
before = len(merged)
merged = merged[(merged["solar_zenith"] <= 85.0) & (merged["ghi"] >= 0)
                & (merged["ghi"].notna()) & (merged["image_exists"])].reset_index(drop=True)
print(f"Quality filter: {before} -> {len(merged)} rows")

# Ramp detection (|ΔGHI| > 50 W/m² in 60s)
dg = merged["ghi"].diff(6).abs() / 1.0
merged["is_ramp"] = (dg > 50.0).fillna(False)
print(f"Ramp events: {int(merged['is_ramp'].sum())} ({merged['is_ramp'].mean()*100:.1f}%)")

# Chronological split (5 train / 1 val / 2 test by date)
dates = sorted(merged["timestamp"].dt.date.unique())
print(f"\nUnique dates: {len(dates)}")
n_tr = max(1, int(len(dates) * 0.625))
n_val = max(1, int(len(dates) * 0.125))
train_dates = set(dates[:n_tr])
val_dates = set(dates[n_tr:n_tr + n_val])
test_dates = set(dates[n_tr + n_val:])

train_df = merged[merged["timestamp"].dt.date.isin(train_dates)].reset_index(drop=True)
val_df   = merged[merged["timestamp"].dt.date.isin(val_dates)].reset_index(drop=True)
test_df  = merged[merged["timestamp"].dt.date.isin(test_dates)].reset_index(drop=True)

SPLITS_DIR = PERSIST_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)
train_df.to_parquet(SPLITS_DIR / "train.parquet")
val_df.to_parquet(SPLITS_DIR / "val.parquet")
test_df.to_parquet(SPLITS_DIR / "test.parquet")

print(f"\nSplit sizes:")
print(f"  train: {len(train_df):>6} rows ({len(train_dates)} days)")
print(f"  val:   {len(val_df):>6} rows ({len(val_dates)} days)")
print(f"  test:  {len(test_df):>6} rows ({len(test_dates)} days)")
print(f"  GHI range overall: [{merged['ghi'].min():.1f}, {merged['ghi'].max():.1f}] W/m²")


## 4. Define CS-VAE

In [ ]:
# ==== CS-VAE model definition ====
import torch
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(32, 64, 128, 256)):
        super().__init__()
        layers, in_ch = [], 3
        for ch in channels:
            layers.extend([
                nn.Conv2d(in_ch, ch, 4, 2, 1),
                nn.GroupNorm(min(32, ch), ch),
                nn.SiLU(inplace=True),
            ])
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc_mu = nn.Linear(channels[-1], latent_dim)
        self.fc_lv = nn.Linear(channels[-1], latent_dim)
    def forward(self, x):
        h = self.pool(self.conv(x)).flatten(1)
        return self.fc_mu(h), self.fc_lv(h)

class Decoder(nn.Module):
    def __init__(self, latent_dim=64, channels=(256, 128, 64, 32)):
        super().__init__()
        self.init_ch = channels[0]
        self.fc = nn.Linear(latent_dim, channels[0] * 8 * 8)
        layers = []
        for i in range(len(channels) - 1):
            layers.extend([
                nn.ConvTranspose2d(channels[i], channels[i+1], 4, 2, 1),
                nn.GroupNorm(min(32, channels[i+1]), channels[i+1]),
                nn.SiLU(inplace=True),
            ])
        layers.extend([nn.ConvTranspose2d(channels[-1], 3, 4, 2, 1), nn.Sigmoid()])
        self.deconv = nn.Sequential(*layers)
    def forward(self, z):
        h = self.fc(z).view(-1, self.init_ch, 8, 8)
        return self.deconv(h)

class CloudStateVAE(nn.Module):
    def __init__(self, latent_dim=64, beta=0.1):
        super().__init__()
        self.latent_dim = latent_dim
        self.beta = beta
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
    def reparam(self, mu, lv):
        return mu + torch.exp(0.5 * lv) * torch.randn_like(mu)
    def forward(self, x):
        mu, lv = self.encoder(x)
        z = self.reparam(mu, lv)
        return self.decoder(z), mu, lv
    def loss(self, x, recon, mu, lv):
        rec = F.mse_loss(recon, x)
        kl = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
        return {"loss": rec + self.beta * kl, "recon": rec, "kl": kl}
    @torch.no_grad()
    def encode_mu(self, x):
        mu, _ = self.encoder(x)
        return mu

n_params = sum(p.numel() for p in CloudStateVAE().parameters())
print(f"CS-VAE parameters: {n_params:,}")


## 5. Image dataset

In [ ]:
# ==== Image Dataset (loads JPEGs on the fly) ====
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import pandas as pd

def load_img(path, size=128):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    side = min(w, h)
    l, t = (w - side) // 2, (h - side) // 2
    img = img.crop((l, t, l + side, t + side)).resize((size, size), Image.BILINEAR)
    return np.array(img, dtype=np.float32) / 255.0

class SkyImageDataset(Dataset):
    def __init__(self, parquet_path, size=128):
        self.df = pd.read_parquet(parquet_path)
        if "image_exists" in self.df.columns:
            self.df = self.df[self.df["image_exists"]].reset_index(drop=True)
        self.size = size
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        p = self.df.iloc[i]["image_path"]
        arr = load_img(p, self.size)
        return torch.from_numpy(arr).permute(2, 0, 1)

for sp in ["train", "val", "test"]:
    ds = SkyImageDataset(SPLITS_DIR / f"{sp}.parquet")
    print(f"  {sp}: {len(ds)} images")


## 6. Train CS-VAE
We use 128×128 images and 20 epochs (trimmed from 100 — empirically sufficient for this dataset size with 2.5M parameters).

In [ ]:
# ==== Train CS-VAE ====
from tqdm import tqdm
import time

IMG_SIZE = 128
LATENT_DIM = 64
BATCH = 32
EPOCHS = 20           # trimmed from 100 — sufficient for 128x128 with this dataset size
LR = 1e-4
SEED = 42

torch.manual_seed(SEED); np.random.seed(SEED)

train_ds = SkyImageDataset(SPLITS_DIR / "train.parquet", size=IMG_SIZE)
val_ds   = SkyImageDataset(SPLITS_DIR / "val.parquet",   size=IMG_SIZE)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

model = CloudStateVAE(latent_dim=LATENT_DIM, beta=0.1).to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

print(f"Training VAE: {EPOCHS} epochs, batch={BATCH}, img={IMG_SIZE}x{IMG_SIZE}, latent_dim={LATENT_DIM}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
print("=" * 70)

best_val = float("inf")
history = []
t_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    tl, tr, tk = 0, 0, 0
    for img in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        img = img.to(DEVICE, non_blocking=True)
        rec, mu, lv = model(img)
        losses = model.loss(img, rec, mu, lv)
        opt.zero_grad(); losses["loss"].backward(); opt.step()
        tl += losses["loss"].item(); tr += losses["recon"].item(); tk += losses["kl"].item()
    tl /= len(train_loader); tr /= len(train_loader); tk /= len(train_loader)

    model.eval()
    vl = 0
    with torch.no_grad():
        for img in val_loader:
            img = img.to(DEVICE, non_blocking=True)
            rec, mu, lv = model(img)
            vl += model.loss(img, rec, mu, lv)["loss"].item()
    vl /= max(len(val_loader), 1)

    elapsed = (time.time() - t_start) / 60
    print(f"Epoch {epoch:3d}/{EPOCHS} | train={tl:.4f} (rec={tr:.4f}, kl={tk:.4f}) | val={vl:.4f} | {elapsed:.1f} min")
    history.append({"epoch": epoch, "train_loss": tl, "train_recon": tr, "train_kl": tk, "val_loss": vl})

    if vl < best_val:
        best_val = vl
        torch.save(model.state_dict(), CHECKPOINT_DIR / "vae_best.pt")
        print(f"  [best] saved checkpoint (val={vl:.4f})")

torch.save(model.state_dict(), CHECKPOINT_DIR / "vae_final.pt")
pd.DataFrame(history).to_csv(RESULTS_DIR / "vae_training_history.csv", index=False)
print("=" * 70)
print(f"VAE training complete. Best val loss: {best_val:.4f}. Total time: {(time.time()-t_start)/60:.1f} min")


## 7. Extract latents and compute CTI
Run frozen VAE over all splits. CTI = L2 norm of variance of latent velocity over a 10-frame window.

In [ ]:
# ==== Extract latents + compute CTI ====
from torch.utils.data import DataLoader

def cti_from_latents(Z, window=10):
    """Compute CTI as L2 norm of variance of latent velocity over a sliding window."""
    T = Z.shape[0]
    cti = np.zeros(T, dtype=np.float32)
    for t in range(window, T):
        win = Z[t - window:t]
        v = np.diff(win, axis=0)
        var = v.var(axis=0)
        cti[t] = np.linalg.norm(var, ord=2)
    return cti

# Load best VAE
model = CloudStateVAE(latent_dim=LATENT_DIM, beta=0.1).to(DEVICE)
model.load_state_dict(torch.load(CHECKPOINT_DIR / "vae_best.pt", map_location=DEVICE))
model.eval()

print("Extracting latents for train / val / test ...")
for split in ["train", "val", "test"]:
    ds = SkyImageDataset(SPLITS_DIR / f"{split}.parquet", size=IMG_SIZE)
    loader = DataLoader(ds, batch_size=64, shuffle=False, num_workers=2)
    all_mu = []
    with torch.no_grad():
        for img in tqdm(loader, desc=f"Encoding {split}"):
            img = img.to(DEVICE, non_blocking=True)
            all_mu.append(model.encode_mu(img).cpu().numpy())
    Z = np.concatenate(all_mu, axis=0).astype(np.float32)
    cti = cti_from_latents(Z, window=10)

    df = ds.df
    ghi = df["ghi"].values.astype(np.float32)
    cov_cols = [c for c in ["solar_zenith", "clear_sky_index", "temperature",
                            "humidity", "wind_speed"] if c in df.columns]
    cov = df[cov_cols].fillna(0).values.astype(np.float32) if cov_cols else np.zeros((len(df), 0), np.float32)

    np.save(LATENT_DIR / f"{split}_latents.npy", Z)
    np.save(LATENT_DIR / f"{split}_cti.npy",     cti)
    np.save(LATENT_DIR / f"{split}_ghi.npy",     ghi)
    np.save(LATENT_DIR / f"{split}_covariates.npy", cov)
    np.save(LATENT_DIR / f"{split}_is_ramp.npy", df["is_ramp"].values.astype(bool))

    print(f"  {split}: Z={Z.shape}, CTI range=[{cti.min():.4f}, {cti.max():.4f}], "
          f"GHI range=[{ghi.min():.1f}, {ghi.max():.1f}], covariates={cov.shape}")
print("Latent extraction complete.")


## 8. Zip outputs and download

In [ ]:
# ==== Zip outputs and prepare download ====
import shutil
zip_path = WORK_DIR / "solarsde_outputs.zip"
if zip_path.exists(): zip_path.unlink()
print(f"Zipping {PERSIST_DIR} -> {zip_path}")
shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", root_dir=PERSIST_DIR)
size_mb = zip_path.stat().st_size / 1e6
print(f"Archive size: {size_mb:.1f} MB")

if IN_COLAB:
    from google.colab import files
    try:
        files.download(str(zip_path))
        print("Download triggered (check browser).")
    except Exception as e:
        print(f"Auto-download unavailable: {e}. Download manually from {zip_path}")
else:
    print(f"Archive at: {zip_path}  — download via Kaggle output tab or file browser.")


## Summary

You should now see in persistent storage:
- `splits/` — train/val/test parquet files
- `checkpoints/vae_best.pt` — trained CS-VAE
- `latents/` — encoded latents + CTI + GHI + covariates for all splits

**Next:** run Notebook 2 — it will read from these files and train the Neural SDE and Score Decoder.


In [ ]:
# ==== Final summary ====
print("=" * 70)
print("NOTEBOOK 1 COMPLETE")
print("=" * 70)
print(f"Persistent storage: {PERSIST_DIR}")
for p in sorted(PERSIST_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(PERSIST_DIR)}: {p.stat().st_size/1e6:.2f} MB")
print()
print("Next: open 02_sde_score_main.ipynb")
